In [ ]:
import os

import boto3
from botocore.exceptions import BotoCoreError, ClientError, NoCredentialsError

# boto3 automatically reads:
# AWS_ACCESS_KEY_ID
# AWS_SECRET_ACCESS_KEY
# Optional: AWS_SESSION_TOKEN

required = ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY")
missing = [name for name in required if not os.getenv(name)]

if missing:
    raise RuntimeError(f"Missing environment variables: {', '.join(missing)}")



In [ ]:

session = boto3.Session(region_name="us-east-1")
s3 = session.client("s3")

try:
    response = s3.list_buckets()

    print("Available S3 buckets:")
    for bucket in response.get("Buckets", []):
        print(f"- {bucket['Name']}")

except NoCredentialsError:
    print("AWS credentials were not found.")
except ClientError as exc:
    print(f"AWS API error: {exc}")
except BotoCoreError as exc:
    print(f"AWS SDK error: {exc}")

# generate RSA key pair, private and public key 
# public key to be uploaed to cloud
# private key must remain in ~/.ssh directory in linux

```
mkdir -p ~/.ssh
chmod 700 ~/.ssh

ssh-keygen \
  -t rsa \
  -b 2048 \
  -m PEM \
  -C "ec2emrkey" \
  -f ~/.ssh/ec2emrkey.pem \
  -N ""

ls ~/.ssh

chmod 600 ~/.ssh/ec2emrkey.pem
chmod 644 ~/.ssh/ec2emrkey.pem.pub
```

In [ ]:
# now upload the public key to AWS

from pathlib import Path
from botocore.exceptions import ClientError

KEY_NAME = "ec2emrkey"
PUBLIC_KEY_PATH = Path.home() / ".ssh" / "ec2emrkey.pem.pub"

ec2 = session.client("ec2")

if not PUBLIC_KEY_PATH.exists():
    raise FileNotFoundError(
        f"Public key not found: {PUBLIC_KEY_PATH}\n"
        "Generate it first using ssh-keygen."
    )

try:
    response = ec2.import_key_pair(
        KeyName=KEY_NAME,
        PublicKeyMaterial=PUBLIC_KEY_PATH.read_bytes()
    )

    print("Key imported successfully")
    print("Key name:", response["KeyName"])
    print("Key fingerprint:", response["KeyFingerprint"])

except ClientError as exc:
    error_code = exc.response["Error"]["Code"]

    if error_code == "InvalidKeyPair.Duplicate":
        print(
            f"A key named {KEY_NAME!r} already exists in "
            f"{session.region_name}; no import was performed."
        )
    else:
        raise

# create ec2 instance

Use the `ec2emerkey` while create a AWS EC2 Instance, it should be of type m4.large, 8 GB RAM, 2 vCPU

in linux/wsl, replace with ubuntu name if other vm type, else leave it as it is for ubuntu vm

replace `PUBLIC_DNS_NAME`
    

`-4` is for ip v4

ssh -4  -i ~/.ssh/ec2emrkey.pem ubuntu@PUBLIC_DNS_NAME


In [ ]:
import boto3
import ipaddress
import urllib.request
from botocore.exceptions import ClientError

# Already established:
# session = boto3.Session(region_name="us-east-1")

region = session.region_name
ec2 = session.client("ec2")
emr = session.client("emr")
iam = session.client("iam")
sts = session.client("sts")

print("Region:", region)
print("Identity:", sts.get_caller_identity()["Arn"])
print("Account:", sts.get_caller_identity()["Account"])

In [ ]:
current_ip = urllib.request.urlopen(
    "https://checkip.amazonaws.com",
    timeout=10
).read().decode().strip()

ipaddress.ip_address(current_ip)
current_cidr = f"{current_ip}/32"

print("Current public IP:", current_ip)
print("Security-group CIDR:", current_cidr)

In [ ]:
vpcs = ec2.describe_vpcs()["Vpcs"]

for vpc in vpcs:
    print(
        "VPC:",
        vpc["VpcId"],
        "CIDR:", vpc["CidrBlock"],
        "Default:", vpc.get("IsDefault", False),
        "State:", vpc["State"]
    )

In [ ]:
default_vpcs = [
    vpc for vpc in vpcs
    if vpc.get("IsDefault", False)
]

if not default_vpcs:
    raise RuntimeError("No default VPC found")

vpc_id = default_vpcs[0]["VpcId"]

print("Selected VPC:", vpc_id)

In [ ]:
def get_subnet_route_table(vpc_id, subnet_id):
    response = ec2.describe_route_tables(
        Filters=[
            {
                "Name": "association.subnet-id",
                "Values": [subnet_id]
            }
        ]
    )

    if response["RouteTables"]:
        return response["RouteTables"][0]

    # Subnet uses the VPC's main route table
    response = ec2.describe_route_tables(
        Filters=[
            {
                "Name": "vpc-id",
                "Values": [vpc_id]
            },
            {
                "Name": "association.main",
                "Values": ["true"]
            }
        ]
    )

    if response["RouteTables"]:
        return response["RouteTables"][0]

    return None


def has_internet_gateway_route(route_table):
    if not route_table:
        return False

    for route in route_table.get("Routes", []):
        if (
            route.get("DestinationCidrBlock") == "0.0.0.0/0"
            and route.get("GatewayId", "").startswith("igw-")
            and route.get("State") == "active"
        ):
            return True

    return False

In [ ]:
subnets = ec2.describe_subnets(
    Filters=[
        {
            "Name": "vpc-id",
            "Values": [vpc_id]
        }
    ]
)["Subnets"]

public_subnets = []

for subnet in subnets:
    route_table = get_subnet_route_table(
        vpc_id,
        subnet["SubnetId"]
    )

    internet_route = has_internet_gateway_route(route_table)
    auto_public_ip = subnet.get("MapPublicIpOnLaunch", False)

    print(
        "Subnet:", subnet["SubnetId"],
        "AZ:", subnet["AvailabilityZone"],
        "CIDR:", subnet["CidrBlock"],
        "Available IPs:", subnet["AvailableIpAddressCount"],
        "Auto public IP:", auto_public_ip,
        "Internet route:", internet_route
    )

    if internet_route and auto_public_ip:
        public_subnets.append(subnet)

In [ ]:
# subnet with most available address

if not public_subnets:
    raise RuntimeError(
        "No public subnet with automatic public-IP assignment was found"
    )

selected_subnet = max(
    public_subnets,
    key=lambda subnet: subnet["AvailableIpAddressCount"]
)

subnet_id = selected_subnet["SubnetId"]

print("Selected subnet:", subnet_id)
print("Availability Zone:", selected_subnet["AvailabilityZone"])

In [ ]:
key_pairs = ec2.describe_key_pairs()["KeyPairs"]

for key in key_pairs:
    print(
        "Key name:", key["KeyName"],
        "Key ID:", key.get("KeyPairId"),
        "Fingerprint:", key.get("KeyFingerprint")
    )

In [ ]:
key_name = "ec2emrkey"

matching_keys = [
    key for key in key_pairs
    if key["KeyName"] == key_name
]

if not matching_keys:
    raise RuntimeError(
        f"EC2 key {key_name!r} does not exist in {region}"
    )

print("Using key:", key_name)

In [ ]:
security_groups = ec2.describe_security_groups(
    Filters=[
        {
            "Name": "vpc-id",
            "Values": [vpc_id]
        }
    ]
)["SecurityGroups"]

for sg in security_groups:
    print(
        "SG ID:", sg["GroupId"],
        "Name:", sg["GroupName"],
        "Description:", sg["Description"]
    )